# mu-logsigma-encoder-head — worked example 3: Add the reparameterization trick to sample z from (mu, logsigma)

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `mu-logsigma-encoder-head`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

The reparameterization trick enables gradients to flow through the sampling operation in a VAE. Instead of sampling `z ~ N(mu, exp(logsigma)^2)` directly (which is not differentiable with respect to mu and logsigma), we compute `z = mu + exp(logsigma) * epsilon` where `epsilon ~ N(0, I)`. This factorizes the stochastic part (epsilon, no parameters) from the deterministic part (mu, logsigma, has parameters).

## Worked solution

**Step 1 — compute the encoder head output.** A double-width linear + chunk gives us `mu` and `logsigma` each of shape `(B, latent_dim)`.

**Step 2 — sample epsilon.** `epsilon = torch.randn_like(mu)` samples from `N(0, I)` with the same shape as `mu`. Using `randn_like` ensures the epsilon is on the same device and dtype as mu.

**Step 3 — apply the reparameterization.** `z = mu + torch.exp(logsigma) * epsilon`. The `exp(logsigma)` term is the standard deviation (we store log-std rather than std directly to keep it unbounded and numerically stable).

**Step 4 — verify shape.** `z` should have the same shape as `mu`: `(B, latent_dim)`. Check this with an assertion.

**Step 5 — verify gradients flow.** Call `z.sum().backward()` and confirm `mu.grad` and `logsigma.grad` are non-None tensors. If the trick is implemented correctly, gradients flow back through both.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

def encoder_head_and_reparameterize(features, weight, bias, latent_dim):
    """
    Full VAE encoder head + reparameterization sample.
    features: (B, D) with requires_grad=True if you want grad through them
    weight: (2*latent_dim, D), bias: (2*latent_dim,)
    Returns (z, mu, logsigma)
    """
    params   = features @ weight.T + bias           # (B, 2*latent_dim)
    mu, logsigma = params.chunk(2, dim=-1)           # each (B, latent_dim)

    # Reparameterization: z = mu + exp(logsigma) * eps
    eps = torch.randn_like(mu)                       # N(0, I), no grad
    z   = mu + torch.exp(logsigma) * eps
    return z, mu, logsigma

# Exercise
torch.manual_seed(13)
B, D, L = 6, 20, 10

features = torch.randn(B, D, requires_grad=True)
weight   = torch.randn(2 * L, D, requires_grad=True)
bias     = torch.randn(2 * L, requires_grad=True)

torch.manual_seed(13)  # reseed so randn_like is deterministic
z, mu, logsigma = encoder_head_and_reparameterize(features, weight, bias, L)

print(f"z shape:        {z.shape}")
print(f"mu shape:       {mu.shape}")
print(f"logsigma shape: {logsigma.shape}")
assert z.shape == (B, L)

# Gradient flows through mu and logsigma
z.sum().backward()
assert weight.grad is not None, "gradient should flow through weight"
assert bias.grad   is not None, "gradient should flow through bias"
print("Gradients flow through reparameterization: OK")